# 04 — NIAH retention curves and the control battery

**Stage:** Proposal Stages 3–4 on a single checkpoint. **Produces:** Table 4 (controls
C1–C4), Table 6 (retention summary), and the raw rows behind Figures 4–7.

### What changed from the pilot

| pilot | here | why |
|---|---|---|
| decoded `ahn_raw` (pre-`o_proj`) | decodes `o_t = o_proj(ahn_raw)` | the pilot's vector was in head-concat space; dims coincide at 3B so it ran and returned noise |
| `vec @ unembed.T` | `readout_logits` with final RMSNorm | Qwen applies `model.norm` before `lm_head` |
| C1 as `o_t(AHN) − o_t(NOWRITE)` ≡ `o_t` | C1 on the **residual stream** | the AHN output is zero under NOWRITE by construction, so the control was vacuous |
| needle at token ~5 | needle placed past `num_attn_sinks` | tokens in the sink prefix are never compressed |
| best layer chosen on the same data it is plotted from | layers fixed in advance | selection-on-test |
| 2 needles (a third silently dropped) | needles filtered up front | cohort size becomes a decision |
| no CIs, no fit diagnostics | bootstrap CIs and R² | Table 6's R² column decides whether "half-life" is even meaningful |

**Prerequisite:** notebook 01 gates pass and `02_table3_jlens_validation.json` says
`TABLE_3_PASSED: true`. If the lens is not validated, run this with `USE_JLENS=False`
to get logit-lens numbers and label every figure "logit lens, preliminary" — that is a
legitimate pilot, but it is not RQ2.


In [17]:
# --- GPU slice: CHANGE THIS EVERY RUN --------------------------------------------
# The H100s are MIG-partitioned: a process gets one ~20 GB slice, not a whole card,
# and the slice UUIDs are regenerated every time the box is recycled (~48h). A bare
# index ("0", "3") does NOT select a MIG slice -- it silently lands somewhere else.
#   nvidia-smi -L    list the slices
#   nvidia-smi       see which are actually idle (four of us share this box)
# Exporting in the shell does not reach the Jupyter kernel, and CUDA reads this once
# at init, so it must be set here -- before anything imports torch.
import os
MIG_UUID = "MIG-802c3ecc-8c66-53d4-9fb5-60712ea8f619"   # <- paste, e.g. "MIG-802c3ecc-8c66-53d4-9fb5-60712ea8f619"
if MIG_UUID:
    os.environ["CUDA_VISIBLE_DEVICES"] = MIG_UUID
elif not os.environ.get("CUDA_VISIBLE_DEVICES"):
    print("! MIG_UUID empty and CUDA_VISIBLE_DEVICES unset -- this kernel lands on "
          "whatever slice it defaults to, possibly one a teammate is using. "
          "Run `nvidia-smi -L`, pick an idle slice, paste its UUID above.")

# --- bootstrap -------------------------------------------------------------------
# `ahn_interp.py` lives at the repo root; this walks up the tree to find it.
# Run this notebook from inside the clone -- nothing needs uploading.
import os, sys, json, importlib

def _find_module(name="ahn_interp.py", depth=4):
    here = os.getcwd()
    for _ in range(depth):
        for cand in (here, os.path.join(here, "src"), os.path.join(here, "notebooks")):
            if os.path.exists(os.path.join(cand, name)):
                return cand
        here = os.path.dirname(here)
    return None

_root = _find_module()
assert _root, ("ahn_interp.py not found. Run this notebook from inside the repo "
               "clone -- `git clone` it on the box rather than copying notebooks "
               "around; the bootstrap searches four levels up from the cwd.")
if _root not in sys.path:
    sys.path.insert(0, _root)

# Pin the working directory to wherever ahn_interp.py actually lives (normally the
# repo root). Without this, relative paths in CFG (results_dir, configs/*.json) resolve
# against whatever directory Jupyter happened to open in -- e.g. running this notebook
# from inside notebooks/ silently writes results to notebooks/results/... instead of
# results/... at the repo root, which is where every other notebook and 05's analysis
# step expect to find them.
os.chdir(_root)

import ahn_interp as ai
importlib.reload(ai)
ai.set_seed()
print("ahn_interp loaded from", _root)
print("working directory pinned to", os.getcwd())


ahn_interp loaded from /home/jupyter-dphs-a263/Interpretability-study-of-Artificial-Hippocampus-Networks
working directory pinned to /home/jupyter-dphs-a263/Interpretability-study-of-Artificial-Hippocampus-Networks


In [18]:
# --- experiment configuration ----------------------------------------------------
# Everything that changes what a number MEANS lives here and gets saved with the run.
CFG = dict(
    model_path      = ai.resolve_ckpt("Qwen-2.5-Instruct-3B-AHN-GDN"),  # via $AHN_CKPT_ROOT, default <repo>/merged_ckpt -- never hardcode /home/...
    cell            = "GatedDeltaNet",     # GatedDeltaNet | DeltaNet | Mamba2
    scale           = "3B",
    sliding_window  = 8064,                # proposal value; upstream eval uses 8064
    num_attn_sinks  = 128,                 # upstream eval default. NOT zero.
    attn_impl       = "flash_attention_2", # "eager" on T4/P100 (no Ampere -> no FA2)
    dtype           = "bfloat16",          # "float16" on T4/P100
    results_dir     = "results/run_3b_gdn",
)
ai.set_results_dir(CFG["results_dir"])
print(json.dumps(CFG, indent=2))


{
  "model_path": "/home/jupyter-dphs-a263/Interpretability-study-of-Artificial-Hippocampus-Networks/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN",
  "cell": "GatedDeltaNet",
  "scale": "3B",
  "sliding_window": 8064,
  "num_attn_sinks": 128,
  "attn_impl": "flash_attention_2",
  "dtype": "bfloat16",
  "results_dir": "results/run_3b_gdn"
}


In [19]:
EXP = dict(
    layers              = [9, 18, 27],          # fixed in advance, matches the J-lens map
    # Prompt length is roughly num_attn_sinks + sliding_window + eviction_distance, so
    # each distance sets the cost of its own conditions: 8192 -> ~16.4K tokens, 16384 ->
    # ~24.6K. Those two dominated the sweep budget. 16384 is dropped from run 1 and added
    # back only if the decay curve has not flattened by 8192 -- six points still support
    # the exponential fit, and Table 6's R2 column is what says whether it does.
    #
    # distance=0 is also dropped: build_niah_prompt puts the needle at ~145 and the
    # compression boundary at n - sliding_window, which for distance=0 lands at ~146, so
    # the ACTUAL eviction distance is ~1 token and the needle_is_evicted check
    # (sinks <= needle_pos < window_start) is one token from failing. Some filler
    # variants would be silently dropped. 64 is the smallest distance that is safely
    # past the boundary for every filler.
    eviction_distances  = [64, 256, 512, 1024, 2048, 4096, 8192],   # add 16384 if needed
    needle_candidates   = ["Paris", "banana", "Tokyo", "violin", "cinnamon",
                           "harbour", "lantern", "sapphire", "meadow", "trumpet"],
    n_filler_variants   = 3,                    # repeats per (needle, distance)
    use_jlens           = True,
    jlens_path          = os.path.join(CFG["results_dir"], "jlens_qwen25_3b.pt"),
)
print(json.dumps(EXP, indent=2))


{
  "layers": [
    9,
    18,
    27
  ],
  "eviction_distances": [
    64,
    256,
    512,
    1024,
    2048,
    4096,
    8192
  ],
  "needle_candidates": [
    "Paris",
    "banana",
    "Tokyo",
    "violin",
    "cinnamon",
    "harbour",
    "lantern",
    "sapphire",
    "meadow",
    "trumpet"
  ],
  "n_filler_variants": 3,
  "use_jlens": true,
  "jlens_path": "results/run_3b_gdn/jlens_qwen25_3b.pt"
}


In [20]:
CFG["model_path"] = ai.resolve_ckpt("Qwen-2.5-Instruct-3B-AHN-GDN")  # via $AHN_CKPT_ROOT, default <repo>/merged_ckpt -- never hardcode /home/...


In [21]:
import torch, numpy as np, time
bundle = ai.load_ahn_model(
    CFG["model_path"], dtype=getattr(torch, CFG["dtype"]),
    attn_implementation=CFG["attn_impl"],
    sliding_window=CFG["sliding_window"], num_attn_sinks=CFG["num_attn_sinks"],
)
tok, probe = bundle.tokenizer, ai.AHNProbe(bundle)

# Loading the lens and checking Table 3 are two separate questions and used to share a
# try/except. That was a hard blocker: TABLE_3_PASSED is currently False (checks 2 and 3
# fail, see notebook 02), the `except` caught FileNotFoundError only, so the AssertionError
# escaped and killed the notebook here -- before a single measurement -- even though
# README "Next steps" item 4 explicitly says to run this WITH the J-lens.
#
# Now: a missing .pt falls back to the logit lens (unchanged behaviour), a missing or
# failing Table 3 is a loud warning that stamps lens_validated=False onto every saved row.
lens = None
LENS_VALIDATED = False

if EXP["use_jlens"]:
    try:
        lens = ai.JacobianLens.load(EXP["jlens_path"], map_location=str(bundle.model.device))
        print("J-lens loaded, layers:", sorted(lens.jacobians))
    except FileNotFoundError:
        print("! no J-lens found at", EXP["jlens_path"])
        print("  Falling back to LOGIT LENS. Label every figure 'logit lens, preliminary'.")
        print("  This is not RQ2.")
        EXP["use_jlens"] = False

if EXP["use_jlens"]:
    try:
        v = ai.load_json("02_table3_jlens_validation.json")
        LENS_VALIDATED = bool(v.get("TABLE_3_PASSED"))
        print("Table 3 passed:", LENS_VALIDATED)
    except FileNotFoundError:
        print("! 02_table3_jlens_validation.json not found in", CFG["results_dir"])
        print("  It was produced on the GPU box by notebook 02 but never downloaded.")
        print("  Proceeding with lens_validated=False.")

    if not LENS_VALIDATED:
        print()
        print("!! PROCEEDING WITH AN UNVALIDATED LENS -- deliberate, not an oversight.")
        print("   Table 3 checks 2 and 3 fail: the J-lens does not reach top-1 on known")
        print("   facts. It does beat the plain logit lens by 8-204x on the same prompts,")
        print("   and RQ2 needs rank SEPARATION between needle and distractor, not top-1.")
        print("   This notebook's control battery (C1/C2/C4) tests that property directly,")
        print("   which is exactly the evidence Gautam asked for before ruling on the lens.")
        print("   Every row is stamped lens_validated=False; label every figure")
        print("   'J-lens, not validated on Table 3' until that ruling lands.")

READOUT = "jlens" if EXP["use_jlens"] else "logit_lens"
EXP["lens_validated"] = LENS_VALIDATED
print()
print("readout:", READOUT, "| lens_validated:", LENS_VALIDATED)


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.02it/s]

! no J-lens found at results/run_3b_gdn/jlens_qwen25_3b.pt
  Falling back to LOGIT LENS. Label every figure 'logit lens, preliminary'.
  This is not RQ2.

readout: logit_lens | lens_validated: False


In [22]:
needles = ai.single_token_needles(tok, EXP["needle_candidates"])
assert len(needles) >= 5, "need at least 5 single-token needles for a usable cohort"

# distractors for control C2: semantically near the needle, absent from the context
DISTRACTORS = {"Paris": "London", "banana": "mango", "Tokyo": "Osaka",
               "violin": "cello", "cinnamon": "nutmeg", "harbour": "wharf",
               "lantern": "torch", "sapphire": "emerald", "meadow": "pasture",
               "trumpet": "clarinet"}
distractor_ids = {}
for n in needles:
    d = DISTRACTORS.get(n)
    ids = tok.encode(f" {d}", add_special_tokens=False) if d else []
    if len(ids) == 1:
        distractor_ids[n] = ids[0]
print(f"{len(distractor_ids)}/{len(needles)} needles have a single-token distractor")


dropped multi-token needles: {'sapphire': 2, 'meadow': 2}
kept 8 single-token needles: ['Paris', 'Tokyo', 'banana', 'cinnamon', 'harbour', 'lantern', 'trumpet', 'violin']
4/8 needles have a single-token distractor


## The measurement

One row per `(needle, eviction distance, filler variant, layer)`. Each row carries
everything Tables 4, 6 and 8 need, plus the four controls, so the whole battery comes
out of one sweep rather than four.


In [23]:
@torch.no_grad()
def measure(needle, needle_id, distance, filler_idx, in_window=False, shuffle=False):
    spec = ai.build_niah_prompt(tok, needle, bundle, eviction_distance=distance,
                                in_window=in_window, filler_idx=filler_idx)
    if not spec["ahn_will_activate"]:
        return []
    if not in_window and not spec["needle_is_evicted"]:
        return []

    prompt = spec["prompt"]
    if shuffle:   # control C3 — destroy word order, keep the token multiset
        w = prompt.split()
        np.random.default_rng(ai.SEED).shuffle(w)
        prompt = " ".join(w)

    ins = tok(prompt, return_tensors="pt").to(bundle.model.device)
    on  = probe.run(ins, nowrite=False, layers=EXP["layers"], capture_residual=True)
    off = probe.run(ins, nowrite=True,  layers=EXP["layers"], capture_residual=True)

    out = []
    for L in EXP["layers"]:
        if L not in on.ahn_raw:
            continue
        o_t = on.o_t(L, pos=-1)
        lg  = ai.readout_logits(o_t, bundle, lens=lens if EXP["use_jlens"] else None, layer=L)

        # C1: residual-stream difference, the non-vacuous zero-state control
        d_res = on.residual(L, pos=-1).float() - off.residual(L, pos=-1).float()
        lg_c1 = ai.readout_logits(d_res, bundle, lens=lens if EXP["use_jlens"] else None, layer=L)

        row = {
            "needle": needle, "layer": L, "readout": READOUT,
            # travels with the data so the caveat cannot be lost between here and
            # a figure caption -- see the lens block above
            "lens_validated": LENS_VALIDATED,
            "requested_distance": distance,
            "eviction_distance": spec["actual_eviction_distance"],
            "in_window": in_window, "shuffled": shuffle, "filler_idx": filler_idx,
            "n_tokens": spec["n_tokens"], "needle_pos": spec["needle_pos"],
            "rank": ai.token_rank(lg, needle_id),
            "p_mem": ai.token_prob(lg, needle_id),
            "entropy": ai.readout_entropy(lg),
            "o_t_norm": float(o_t.float().norm()),
            "rank_c1_residual": ai.token_rank(lg_c1, needle_id),
            "p_mem_c1_residual": ai.token_prob(lg_c1, needle_id),
        }
        if needle in distractor_ids:      # C2
            row["rank_distractor"] = ai.token_rank(lg, distractor_ids[needle])
            row["p_distractor"] = ai.token_prob(lg, distractor_ids[needle])
        out.append(row)
    return out


In [24]:
rows, t0 = [], time.time()
total = len(needles) * len(EXP["eviction_distances"]) * EXP["n_filler_variants"]
done = 0
for needle, nid in needles.items():
    for dist in EXP["eviction_distances"]:
        for fi in range(EXP["n_filler_variants"]):
            rows += measure(needle, nid, dist, fi)
            done += 1
            if done % 20 == 0:
                print(f"[{done}/{total}] {len(rows)} rows, {(time.time()-t0)/60:.1f} min")
            ai.free_cuda()
print(f"main sweep: {len(rows)} rows in {(time.time()-t0)/60:.1f} min")


[20/168] 60 rows, 1.1 min
[40/168] 120 rows, 2.1 min
[60/168] 180 rows, 3.1 min
[80/168] 240 rows, 4.2 min
[100/168] 300 rows, 5.2 min
[120/168] 360 rows, 6.3 min
[140/168] 420 rows, 7.3 min
[160/168] 480 rows, 8.3 min
main sweep: 504 rows in 8.8 min


In [25]:
# C4 pre-eviction baseline (the ceiling) and C3 shuffled context
ctrl_rows = []
for needle, nid in needles.items():
    for fi in range(EXP["n_filler_variants"]):
        ctrl_rows += measure(needle, nid, 0, fi, in_window=True)          # C4 ceiling
    for dist in (1024, 4096):
        ctrl_rows += measure(needle, nid, dist, 0, shuffle=True)          # C3
    ai.free_cuda()
print(f"control rows: {len(ctrl_rows)}")

all_rows = rows + ctrl_rows
ai.save_json({"rows": all_rows, "cfg": CFG, "exp": EXP,
              "needles": needles, "distractors": distractor_ids},
             "04_retention_rows.json")
print("saved -> 04_retention_rows.json  (this is the file notebook 05 reads)")


control rows: 120
saved -> 04_retention_rows.json  (this is the file notebook 05 reads)


## Table 4 — the control battery, evaluated

C1 and C4 have to pass before Table 6 is filled in. C3 failing is *not* a bug — the
Expected-Tables document flags it as potentially the most publishable result in the
project: if shuffling the context barely changes retention, AHN is closer to a learned
recency mechanism than to content memory, which contradicts the framing of the original
AHN paper.


In [26]:
import numpy as np
V = bundle.vocab
def sel(**kw):
    out = all_rows
    for k, v in kw.items():
        out = [r for r in out if r.get(k) == v]
    return out

main   = [r for r in all_rows if not r["in_window"] and not r["shuffled"]]
inwin  = [r for r in all_rows if r["in_window"]]
shuf   = [r for r in all_rows if r["shuffled"]]

T4 = {}

# C1 — the memory's contribution must beat what the residual difference alone explains,
#      and both must beat chance.
T4["C1_zero_state"] = {
    "mean_rank_o_t": float(np.mean([r["rank"] for r in main])),
    "mean_rank_residual_delta": float(np.mean([r["rank_c1_residual"] for r in main])),
    "chance_rank": V / 2,
    "passed": bool(np.mean([r["rank"] for r in main]) < V / 10),
    "note": "fails if the target is at chance in the memory readout: nothing is retained, "
            "or the readout is still in the wrong basis",
}

# C2 — the true needle must beat a semantically near absent token by >= 1 order of magnitude
withd = [r for r in main if "p_distractor" in r]
if withd:
    ratio = float(np.median([(r["p_mem"] + 1e-12) / (r["p_distractor"] + 1e-12) for r in withd]))
    T4["C2_distractor"] = {"median_prob_ratio": ratio, "n": len(withd),
                           "passed": bool(ratio >= 10.0),
                           "note": "below 10x: the readout reflects topic, not the stored item; "
                                   "RQ2 weakens to 'semantic gist'"}

# C3 — shuffling should hurt retention if the state stores content rather than recency
if shuf:
    T4["C3_shuffled_context"] = {
        "mean_rank_ordered": float(np.mean([r["rank"] for r in main
                                            if r["requested_distance"] in (1024, 4096)])),
        "mean_rank_shuffled": float(np.mean([r["rank"] for r in shuf])),
        "passed": bool(np.mean([r["rank"] for r in shuf])
                       > np.mean([r["rank"] for r in main
                                  if r["requested_distance"] in (1024, 4096)])),
        "note": "FAILURE HERE IS A FINDING, not a bug — see Expected_Tables_and_Figures §3",
    }

# C4 — pre-eviction ceiling must be BETTER than any evicted condition.
#      In the pilot it was worse (Paris baseline rank 110712 vs ~95000 evicted), which
#      is the single clearest sign the measurement was not measuring retention.
if inwin:
    T4["C4_pre_eviction_baseline"] = {
        "mean_rank_in_window": float(np.mean([r["rank"] for r in inwin])),
        "mean_rank_evicted": float(np.mean([r["rank"] for r in main])),
        "passed": bool(np.mean([r["rank"] for r in inwin])
                       < np.mean([r["rank"] for r in main])),
        "note": "if the in-window ceiling is worse than the evicted condition, the "
                "placement or the readout is wrong. Stop and fix before Table 6.",
    }

T4["BATTERY_PASSED"] = bool(T4["C1_zero_state"]["passed"]
                            and T4.get("C4_pre_eviction_baseline", {}).get("passed", True))
ai.save_json(T4, "04_table4_controls.json")
print(json.dumps(T4, indent=2))
print("\nC1+C4:", "PASS -> Table 6 may be populated" if T4["BATTERY_PASSED"]
      else "FAIL -> fix instrumentation, do NOT report Table 6")


{
  "C1_zero_state": {
    "mean_rank_o_t": 81598.64087301587,
    "mean_rank_residual_delta": 84048.34523809524,
    "chance_rank": 75968.0,
    "passed": false,
    "note": "fails if the target is at chance in the memory readout: nothing is retained, or the readout is still in the wrong basis"
  },
  "C2_distractor": {
    "median_prob_ratio": 1.0630428401854164,
    "n": 252,
    "passed": false,
    "note": "below 10x: the readout reflects topic, not the stored item; RQ2 weakens to 'semantic gist'"
  },
  "C3_shuffled_context": {
    "mean_rank_ordered": 82470.01388888889,
    "mean_rank_shuffled": 89599.85416666667,
    "passed": true,
    "note": "FAILURE HERE IS A FINDING, not a bug \u2014 see Expected_Tables_and_Figures \u00a73"
  },
  "C4_pre_eviction_baseline": {
    "mean_rank_in_window": 72410.08333333333,
    "mean_rank_evicted": 81598.64087301587,
    "passed": true,
    "note": "if the in-window ceiling is worse than the evicted condition, the placement or the readout is

## Adding more metrics

In [27]:
import json, numpy as np
data = json.load(open("results/run_3b_gdn/04_retention_rows.json"))
rows = data["rows"]
main = [r for r in rows if not r["in_window"] and not r["shuffled"]]

for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L]
    print(f"layer {L}: mean_rank={np.mean([r['rank'] for r in rs]):.0f}  "
          f"mean_p_mem={np.mean([r['p_mem'] for r in rs]):.3e}")

# raw p_mem / p_distractor pairs, unrounded, no epsilon
withd = [r for r in main if "p_distractor" in r][:10]
for r in withd:
    print(r["needle"], r["layer"], r["eviction_distance"],
          "p_mem=", r["p_mem"], "p_distractor=", r["p_distractor"])

layer 9: mean_rank=95556  mean_p_mem=4.731e-08
layer 18: mean_rank=66936  mean_p_mem=2.280e-06
layer 27: mean_rank=82304  mean_p_mem=3.484e-07
Paris 9 80 p_mem= 3.0604149969803984e-07 p_distractor= 4.426898314591199e-08
Paris 18 80 p_mem= 3.4029350448605555e-09 p_distractor= 2.975177659791939e-09
Paris 27 80 p_mem= 1.2797485737792158e-07 p_distractor= 3.667404868679114e-08
Paris 9 87 p_mem= 6.49176570277632e-07 p_distractor= 1.9929214545300056e-07
Paris 18 87 p_mem= 2.542383525927505e-10 p_distractor= 3.8547512404285555e-10
Paris 27 87 p_mem= 1.3731299652874895e-08 p_distractor= 4.6979455881057675e-09
Paris 9 84 p_mem= 6.192524892867368e-08 p_distractor= 3.841699935946963e-07
Paris 18 84 p_mem= 4.31692726010624e-09 p_distractor= 8.843654281109892e-11
Paris 27 84 p_mem= 7.901812892896487e-08 p_distractor= 3.500110423715341e-08
Paris 9 272 p_mem= 4.3821336248583975e-07 p_distractor= 3.3537922661253106e-08


In [28]:
import json, numpy as np

data = json.load(open("results/run_3b_gdn/04_retention_rows.json"))
rows = data["rows"]
main = [r for r in rows if not r["in_window"] and not r["shuffled"]]
V = 151936
EPS = 1e-30  # small enough not to swamp probabilities down to ~1e-24

print("=== C1 per layer (pass bar: mean_rank < %d) ===" % (V // 10))
for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L]
    mr = np.mean([r["rank"] for r in rs])
    print(f"layer {L}: n={len(rs)} mean_rank={mr:.0f}  passes={mr < V/10}")

print("\n=== C2 per layer, corrected epsilon (pass bar: median ratio >= 10) ===")
for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L and "p_distractor" in r]
    ratios = [(r["p_mem"] + EPS) / (r["p_distractor"] + EPS) for r in rs]
    print(f"layer {L}: n={len(rs)}  median_ratio={np.median(ratios):.3f}  "
          f"frac_needle>distractor={np.mean([r>1 for r in ratios]):.2f}  "
          f"frac_pass_10x={np.mean([r>=10 for r in ratios]):.2f}")

=== C1 per layer (pass bar: mean_rank < 15193) ===
layer 9: n=168 mean_rank=95556  passes=False
layer 18: n=168 mean_rank=66936  passes=False
layer 27: n=168 mean_rank=82304  passes=False

=== C2 per layer, corrected epsilon (pass bar: median ratio >= 10) ===
layer 9: n=84  median_ratio=0.347  frac_needle>distractor=0.35  frac_pass_10x=0.14
layer 18: n=84  median_ratio=1.173  frac_needle>distractor=0.55  frac_pass_10x=0.25
layer 27: n=84  median_ratio=3.391  frac_needle>distractor=0.64  frac_pass_10x=0.17


### Gate for this notebook

- [ ] C1 passes (target well inside the top decile of vocabulary, not at chance)
- [ ] C4 passes (in-window ceiling beats every evicted condition)
- [ ] C2 recorded — if the ratio is under 10×, RQ2's claim weakens to "semantic gist"
- [ ] C3 recorded — **if it fails, message Gautam before doing anything else**
- [ ] `04_retention_rows.json` saved

Analysis and figures are in **05_analysis_and_figures.ipynb**, which runs on CPU. Download
`04_retention_rows.json` and run 05 on your laptop — GPU time is the scarce resource,
analysis time is not.


## RULER NIAH cohort — the primary RQ2 cohort, wired up 7 Sep

Expected Tables row 1 of Table 2 makes RULER NIAH n=60 the PRIMARY RQ2 cohort — "ground
truth y* is known by construction" is the proposal's number-one reason-to-believe. Every
NIAH result above uses the homemade `build_niah_prompt` instead: 8 single-token needles x
3 filler variants x 7 distances. `ai.load_ruler()` has existed since this notebook's first
sweep and was never called.

This is the cheapest direct test of candidate (b) from Next steps item 5:
`build_niah_prompt` ends at `"What was the special word?"` with no answer prefix;
`load_ruler` appends RULER's own `answer_prefix`. The readout stays at `pos=-1` in both, so
prompt construction is the only variable that changes between this sweep and the one above.

**What this is not:** a replacement for C1–C4. RULER gives one prompt and one gold answer
per example, not a needle/distractor pair or a way to shuffle just the needle region, so
there is no C2, C3 or C4 analog here — only a C1-shaped question (does the readout find the
real answer above chance in a real cohort). That is exactly the question that matters for
candidate (b), and nothing more should be claimed for it.

In [29]:
# config="8192" looked right by construction (sliding_window + num_attn_sinks = 8192)
# but was WRONG in practice: measured on this box, 7 Sep --
#   config=4096   n_tokens min=3675  median=3681  max=3687
#   config=8192   n_tokens min=7875  median=7881  max=7887   <- under 8192, 60/60 skipped
#   config=16384  n_tokens min=15678 median=15679 max=15684
# simonjegou/ruler's length buckets were built with a different tokenizer than Qwen's;
# Qwen's larger vocab (151,936) compresses English text into noticeably fewer tokens, so
# a bucket built to be "8192 tokens" comes out under 8192 once retokenized here. Only
# 4096/8192/16384 exist as configs for this dataset -- there is no 32768 to fall back to.
# 16384 is the only bucket that clears window+sinks, with ~7,500 tokens of margin and a
# 6-token spread across examples, so it should activate for very close to all 60.
RULER_CFG = dict(config="16384", split="test", n=60, seed=ai.SEED)
ruler_examples = ai.load_ruler(**RULER_CFG)
print(f"loaded {len(ruler_examples)} RULER NIAH examples, config={RULER_CFG['config']}")
print("--- tail of example 0, sanity check the prompt ends where expected ---")
print(ruler_examples[0]["prompt"][-300:])
print("gold answer:", ruler_examples[0]["answer"])

loaded 60 RULER NIAH examples, config=16384
--- tail of example 0, sanity check the prompt ends where expected ---
blue. The sun is yellow. Here we go. There and back again.
The grass is green. The sky is blue. The sun is yellow. Here we go. There and back again.
What is the special magic number for solid-few mentioned in the provided text? The special magic number for solid-few mentioned in the provided text is
gold answer: 7700828


In [30]:
@torch.no_grad()
def measure_ruler(example):
    prompt, answer = example["prompt"], example["answer"]
    # Gold answer's FIRST token only, same convention the RQ3 join already uses for the
    # same reason: the readout is one next-token distribution, not a generation. The
    # leading space matters for Qwen's BPE -- most words tokenize differently with vs.
    # without it, and answer_prefix already ends without a trailing space.
    target_id = tok.encode(" " + answer.lstrip(), add_special_tokens=False)[0]

    ins = tok(prompt, return_tensors="pt").to(bundle.model.device)
    n_tokens = int(ins["input_ids"].shape[1])
    window = bundle.sliding_window or 0
    sinks = bundle.num_attn_sinks or 0
    if not (n_tokens > window + sinks):
        return None   # AHN never activates at this length -- same check build_niah_prompt uses

    # 16384-config prompts run ~1.7x longer than anything else in this notebook, on a
    # 20 GB MIG slice shared with 3 other people (already OOM'd once today on a slice
    # someone else was using). A transient OOM here should not lose the whole 60-example
    # sweep -- skip the example, log why, and keep going.
    try:
        on  = probe.run(ins, nowrite=False, layers=EXP["layers"], capture_residual=True)
        off = probe.run(ins, nowrite=True,  layers=EXP["layers"], capture_residual=True)
    except torch.cuda.OutOfMemoryError as e:
        ai.free_cuda()
        print(f"  ! OOM at n_tokens={n_tokens}, skipping this example: {e}")
        return None

    out = []
    for L in EXP["layers"]:
        if L not in on.ahn_raw:
            continue
        o_t = on.o_t(L, pos=-1)
        lg  = ai.readout_logits(o_t, bundle, lens=lens if EXP["use_jlens"] else None, layer=L)

        d_res = on.residual(L, pos=-1).float() - off.residual(L, pos=-1).float()
        lg_c1 = ai.readout_logits(d_res, bundle, lens=lens if EXP["use_jlens"] else None, layer=L)

        out.append({
            "task": example["task"], "layer": L, "readout": READOUT,
            "lens_validated": LENS_VALIDATED,
            "cohort": "ruler_niah", "ruler_config": RULER_CFG["config"],
            "n_tokens": n_tokens,
            "compression_boundary": n_tokens - window,
            "rank": ai.token_rank(lg, target_id),
            "p_mem": ai.token_prob(lg, target_id),
            "entropy": ai.readout_entropy(lg),
            "rank_c1_residual": ai.token_rank(lg_c1, target_id),
            "p_mem_c1_residual": ai.token_prob(lg_c1, target_id),
        })
    return out

In [31]:
ruler_rows, skipped, t0 = [], 0, time.time()
for i, ex in enumerate(ruler_examples):
    res = measure_ruler(ex)
    if res is None:
        skipped += 1
    else:
        ruler_rows += res
    if (i + 1) % 10 == 0:
        print(f"[{i+1}/{len(ruler_examples)}] {len(ruler_rows)} rows, {skipped} skipped "
              f"(too short for AHN to activate), {(time.time()-t0)/60:.1f} min")
    ai.free_cuda()

print(f"RULER sweep: {len(ruler_rows)} rows from {len(ruler_examples) - skipped}/"
      f"{len(ruler_examples)} examples in {(time.time()-t0)/60:.1f} min")

if skipped > len(ruler_examples) // 2:
    print("! more than half the cohort skipped -- config='8192' is landing short of "
          "window+sinks for most examples. Recheck RULER_CFG before trusting the rows below.")

ai.save_json({"rows": ruler_rows, "cfg": CFG, "exp": EXP, "ruler_cfg": RULER_CFG},
             "04g_ruler_niah_rows.json")
print("saved -> 04g_ruler_niah_rows.json")

[10/60] 30 rows, 0 skipped (too short for AHN to activate), 0.6 min
[20/60] 60 rows, 0 skipped (too short for AHN to activate), 1.3 min
[30/60] 90 rows, 0 skipped (too short for AHN to activate), 1.9 min
[40/60] 120 rows, 0 skipped (too short for AHN to activate), 2.5 min
[50/60] 150 rows, 0 skipped (too short for AHN to activate), 3.1 min
[60/60] 180 rows, 0 skipped (too short for AHN to activate), 3.8 min
RULER sweep: 180 rows from 60/60 examples in 3.8 min
saved -> 04g_ruler_niah_rows.json


### Gate for this section

- [ ] `skipped` is a small fraction of 60 -- most examples actually reach AHN activation
      at config="8192"
- [ ] median `rank` well below chance (~76,000) would be the first real-cohort signal in
      candidate (b)'s favour; at or above chance keeps (b) alive as the explanation
- [ ] compare against `04_retention_rows.json` rows at a matched eviction distance before
      drawing any conclusion -- one cohort's median against the other's is the whole point